# FINESSE — QLoRA fine-tune (free Colab T4)

Runtime → **Change runtime type → T4 GPU**. Then run cells top to bottom.

1. Install deps · 2. Upload a `finesse_sft*.jsonl` (from `training/data/`) · 3. Pick base+params · 4. Train · 5. Download the adapter.

T4 has no bf16, so this notebook uses **fp16** 4-bit QLoRA. Start with the **unified** `finesse_sft.jsonl`; switch to a per-lane file once that lane has ~200+ examples.


## 1. Install


In [ ]:
!pip -q install "transformers>=4.45,<4.48" "trl>=0.12,<0.14" "peft>=0.13" \
    "bitsandbytes>=0.43" "accelerate>=0.34" "datasets>=2.20" sentencepiece
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


## 2. Upload your dataset
Upload one file from `FINESSE/training/data/` — e.g. `finesse_sft.jsonl` (unified) or `finesse_sft_coding.jsonl` (a lane).


In [ ]:
from google.colab import files
up = files.upload()
DATA_FILE = list(up.keys())[0]
print('using', DATA_FILE, '-', sum(1 for _ in open(DATA_FILE)), 'examples')


## 3. Choose base model + hyper-params
`BASE` can be any of your three: `Qwen/Qwen2.5-7B-Instruct`, `meta-llama/Llama-3.1-8B-Instruct`, `Qwen/Qwen2.5-3B-Instruct`.
Re-run cells 3-5 per base to make 'all of them'. (Llama needs a HF token + license acceptance.)


In [ ]:
BASE     = 'Qwen/Qwen2.5-7B-Instruct'   # <- swap to train the others
EPOCHS   = 3
R, ALPHA = 16, 32
MAX_SEQ  = 1024
OUT      = 'finesse-qlora-' + BASE.split('/')[-1].lower() + '-' + DATA_FILE.replace('finesse_sft','').replace('.jsonl','').strip('_') or 'unified'
OUT      = OUT.rstrip('-'); print('adapter ->', OUT)


## 4. Train (QLoRA, 4-bit nf4, fp16 for T4)


In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb,
    device_map='auto', torch_dtype=torch.float16)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(r=R, lora_alpha=ALPHA, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
ds = load_dataset('json', data_files=DATA_FILE, split='train')
def fmt(b): return [tok.apply_chat_template(m, tokenize=False) for m in b['messages']]

cfg = SFTConfig(output_dir=OUT, num_train_epochs=EPOCHS, per_device_train_batch_size=1,
    gradient_accumulation_steps=8, learning_rate=2e-4, warmup_ratio=0.03,
    lr_scheduler_type='cosine', logging_steps=5, save_strategy='epoch', fp16=True,
    optim='paged_adamw_8bit', max_seq_length=MAX_SEQ, gradient_checkpointing=True, report_to='none')
trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds, peft_config=lora,
    formatting_func=fmt, processing_class=tok)
trainer.train()
trainer.save_model(OUT); tok.save_pretrained(OUT)
print('saved adapter ->', OUT)


## 5. Download the adapter
Unzip into `FINESSE/training/adapters/`, then `merge_adapter.py` + serve with vLLM and point `.env LOCAL_LLM_*` at it.


In [ ]:
import shutil; shutil.make_archive(OUT, 'zip', OUT)
from google.colab import files; files.download(OUT + '.zip')
